# ESP32 species classifier v4.1-CV: cross-validated by visit

v4.1 used a fixed validation split of 26 visit tracks to choose the model and set the auto-label thresholds. That's too few: the choice of model came down to noise, and the thresholds rested on 5–8 visits.

This version keeps **the same test visits** (same seed and rule), so results compare directly with v4.1, and replaces the validation split with **K-fold cross-validation by visit**:

1. Every non-test visit goes into one of `FOLDS` folds, assigned per species, so each fold gets a share of every species.
2. Each model arm is trained `FOLDS` times, each time on the other folds, and predicts the fold it didn't see. Together these out-of-fold predictions cover **every non-test visit** (about 120 tracks instead of 26).
3. Training uses a **fixed schedule** (no early stopping), so no checkpoint is chosen by peeking at the held-out fold and the out-of-fold numbers stay honest.
4. The arm and the auto-label thresholds are chosen on the out-of-fold visit tracks. Each threshold needs at least `MIN_AUTO_TRACKS` auto-labelled tracks at `AUTO_TARGET_PRECISION`, and its Wilson 95% lower bound is reported.
5. The chosen arm is retrained on **all** non-test visits, scored once on test, and exported with a hub-ready `species_contract.json`.

Run time is about 1.5 h for 4 folds × 2 arms plus the final model. Set `SPECIES_ARMS = ("esp32_only",)` to halve it.


## 1. Configuration


In [1]:
# ---- Data -------------------------------------------------------------------
DATASET_ROOT = r"D:\Datasets\ESP32Cam_confirmed"
ANNOTATIONS_CSV = DATASET_ROOT + r"\bird_annotations.csv"
INVENTORY_CSV = DATASET_ROOT + r"\ESP32Cam_confirmed_manifest.csv"
PROJECT_ROOT = r"C:\Users\Joe\Desktop\ESP_Classifier\esp32_Classifier"
MODELS_DIR = PROJECT_ROOT + r"\Models"
OUTPUT_ROOT = MODELS_DIR + r"\runs"
OUTSIDE_MANIFEST_DIRS = [DATASET_ROOT + r"\Manifests", MODELS_DIR + r"\Manifests"]
INAT_IMAGE_ROOTS = [r"D:\Datasets\iNaturalist", r"D:\Datasets\Animals", r"E:\Datasets\iNaturalist", r"E:\Datasets\Animals"]
CUB_IMAGE_ROOTS = [r"D:\Datasets\CUB_200\CUB_200_2011\images", r"D:\Datasets\CUB_200_2011\CUB_200_2011\images",
                   r"E:\Datasets\CUB_200\CUB_200_2011\images", r"E:\Datasets\CUB_200_2011\CUB_200_2011\images"]
# Localisers: both are run on every capture; LOCALIZER = "auto" keeps whichever finds more labelled birds.
#  - pi_detector:   the hub's current detector, on the Pi-selected crop, decoded by server/bird_inference.py
#  - esp32_roi_v3:  the v3 ESP32 ROI detector (128 px, int8), on a square around the Pi-selected crop
SERVER_DIR = PROJECT_ROOT + r"\server"
DETECTOR_MODEL = MODELS_DIR + r"\pi_detector_float_stride16_224\bird_detector_motion_crop_stride16_float32.tflite"
ROI_V3_DIR = MODELS_DIR + r"\Multi_Object_Detector\runs\esp32_roi_detector_v3_128_20261110T052149Z\selected_model"
LOCALIZER = "auto"                 # "auto", "esp32_roi_v3" or "pi_detector"
DETECTOR_CACHE_DIR = OUTPUT_ROOT + r"\_detector_cache"
PREVIOUS_SPECIES_DIR = MODELS_DIR + r"\runs\esp32_species_v4_1_20261119T155021Z\esp32_only"  # v4.1 camera-only, scored on the same test crops

SEED = 1234
QUICK_RUN = False
WEIGHTS = "imagenet"

# ---- Visits, test split and cross-validation --------------------------------------
VISIT_GAP_SECONDS = 120            # same camera, captures closer than this belong to one visit
TEST_VISIT_FRACTION = 0.20         # per species, by visit; with SPLIT_SEED this reproduces v4.1's test visits
V41_VAL_FRACTION = 0.15            # only used to reproduce v4.1's test split exactly; these visits join the CV pool
FOLDS = 4                          # cross-validation folds over every non-test visit
FOLD_SEED = 77

# ---- Detector pass -------------------------------------------------------------
DETECTOR_MIN_SCORE = 0.05          # isBird is the gate; the detector only has to localise
DETECTOR_MATCH_IOU = 0.30          # a detection belongs to the labelled box it overlaps most, if IoU >= this

# ---- Species crops -------------------------------------------------------------
PI_MARGIN_FRACTION = 0.20          # server/motion_crop.py static-ROI crop
PI_MIN_CROP = 160
SPECIES_IMG = 224
SPECIES_ALPHA = 1.0
SPECIES_CONTEXT = 1.5              # deployed rule: square crop, side = 1.5 x the box's long side
SPECIES_MIN_CROP = 48
TRAIN_CONTEXT_RANGE = (1.25, 2.2)  # bird fills 45-80% of the crop side
TRAIN_SHIFT = 0.08                 # crop-centre jitter, fraction of crop side
TRAIN_BOX_JITTER = 0.12            # hand-box jitter (scale and shift) to mimic detector error
LOOSE_BIRD_FRACTION = (0.12, 0.45) # fallback crops, as small as the bird in a static-ROI crop
CROP_MIX = {"detector": 0.45, "gt_box": 0.35, "loose": 0.20}
REGIONAL_EXTRA_SPECIES = [
    "American Goldfinch", "House Finch", "Dark-eyed Junco", "White-throated Sparrow", "Carolina Wren",
    "European Starling", "Common Grackle", "Red-winged Blackbird", "Hairy Woodpecker", "Red-breasted Nuthatch",
    "Brown-headed Cowbird", "American Robin", "Northern Mockingbird", "Chipping Sparrow", "Purple Finch",
    "American Tree Sparrow", "Northern Flicker", "Pine Siskin", "Rock Pigeon", "Gray Catbird",
]
SPECIES_GROUPS = {
    "sparrow-like": ["House Sparrow", "Song Sparrow", "White-throated Sparrow", "Chipping Sparrow", "American Tree Sparrow",
                     "Dark-eyed Junco", "House Finch", "Purple Finch", "American Goldfinch", "Pine Siskin"],
    "dove/pigeon": ["Mourning Dove", "Rock Pigeon"],
    "jay/crow": ["Blue Jay", "American Crow"],
    "chickadee/titmouse": ["Black-capped Chickadee", "Tufted Titmouse"],
    "nuthatch": ["White-breasted Nuthatch", "Red-breasted Nuthatch"],
    "woodpecker": ["Downy Woodpecker", "Hairy Woodpecker", "Red-bellied Woodpecker", "Northern Flicker"],
    "cardinal": ["Northern Cardinal"],
    "blackbird-like": ["Common Grackle", "Red-winged Blackbird", "Brown-headed Cowbird", "European Starling"],
    "other songbird": ["Carolina Wren", "American Robin", "Northern Mockingbird", "Gray Catbird"],
}
SPECIES_ESP32_SHARE = 0.65
SPECIES_CLASS_BALANCE_POWER = 0.5  # per-class weight ~ (training visits) ** power
SPECIES_ARMS = ("esp32_plus_outside", "esp32_only")
OUTSIDE_INAT_MIN_SCORE = 0.90

# ---- Automation ----------------------------------------------------------------
AUTO_TARGET_PRECISION = 0.95       # auto-labelled out-of-fold visit tracks must be at least this precise
MIN_AUTO_TRACKS = 10               # ... and a threshold needs at least this many auto-labelled tracks
MIN_TRAIN_VISITS_FOR_AUTO = 5      # species with fewer camera training visits always go to review
MIN_DAYS_FOR_AUTO = 2              # ... and so do species seen on only one day: one-day species teach the model
                                   # that day's scene, not the bird (v4.1-CV: Oct 7 legacy -> "Song Sparrow")
REVIEW_DISAGREEMENT_MIN_PROB = 0.5 # review_queue: model's other answer at least this confident

# ---- Training (fixed schedule: no early stopping, so out-of-fold scores stay honest) ----
BATCH_SIZE = 32
EXAMPLES_PER_EPOCH = 9600
HEAD_EPOCHS = 3
FINE_TUNE_EPOCHS = 15              # v4.1's best checkpoints came after 4-14 fine-tune epochs; cosine decays to ~0
HEAD_LR = 1e-3
FINE_TUNE_PEAK_LR = 2e-4
WEIGHT_DECAY = 1e-4
FINE_TUNE_FROM_BLOCK = "block_6_expand"
LABEL_SMOOTHING = 0.05
REPRESENTATIVE_SAMPLES = 300
TFLITE_THREADS = 4

if QUICK_RUN:
    EXAMPLES_PER_EPOCH, HEAD_EPOCHS, FINE_TUNE_EPOCHS, FOLDS = 256, 1, 1, 2
    REPRESENTATIVE_SAMPLES, MIN_AUTO_TRACKS = 16, 2


## 2. Environment and helpers


In [2]:
import hashlib, io, json, math, os, platform, random, re, time
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath, PureWindowsPath

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageOps

keras = tf.keras
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)


def portable_path(raw):
    """Windows path <-> WSL /mnt/<drive> path, whichever this kernel needs."""
    raw = str(raw).strip()
    win = PureWindowsPath(raw)
    if win.drive:
        if os.name == "nt":
            return Path(raw)
        return Path("/mnt") / win.drive.rstrip(":").lower() / Path(*win.parts[1:])
    m = re.match(r"^/mnt/([A-Za-z])/(.*)$", raw.replace("\\", "/"))
    if m and os.name == "nt":
        return Path(f"{m.group(1).upper()}:/") / Path(*PurePosixPath(m.group(2)).parts)
    return Path(raw)


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def json_safe(v):
    if isinstance(v, dict):
        return {str(k): json_safe(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)):
        return [json_safe(x) for x in v]
    if isinstance(v, Path):
        return str(v)
    if isinstance(v, np.ndarray):
        return json_safe(v.tolist())
    if isinstance(v, np.generic):
        v = v.item()
    if isinstance(v, float) and not math.isfinite(v):
        return None
    return v


def write_json(path, data):
    Path(path).write_text(json.dumps(json_safe(data), indent=2, allow_nan=False), encoding="utf-8")


def first_existing(paths, kind="dir"):
    for p in paths:
        q = portable_path(p)
        if (q.is_dir() if kind == "dir" else q.is_file()):
            return q
    return None


DATASET_ROOT_PATH = portable_path(DATASET_ROOT)
created_utc = datetime.now(timezone.utc)
RUN_NAME = f"esp32_species_v4_1_cv_{created_utc:%Y%m%dT%H%M%SZ}" + ("_quick" if QUICK_RUN else "")
RUN_DIR = portable_path(OUTPUT_ROOT) / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Python", platform.python_version(), "| TensorFlow", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Run directory:", RUN_DIR)


I0000 00:00:1795107485.519244   22126 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795107485.571990   22126 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795107487.267685   22126 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Python 3.12.15 | TensorFlow 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Run directory: /mnt/c/Users/Joe/Desktop/ESP_Classifier/esp32_Classifier/Models/runs/esp32_species_v4_1_cv_20261119T165807Z


## 3. Load annotations, inventory and capture sidecars

One record per image: boxes with species, plus camera ID, sensor, static ROI and deployment from the sidecar JSON. Images whose SHA-256 does not match the inventory are rejected.


In [3]:
ann = pd.read_csv(portable_path(ANNOTATIONS_CSV), encoding="utf-8-sig", low_memory=False)
inv = pd.read_csv(portable_path(INVENTORY_CSV), encoding="utf-8-sig", low_memory=False)
for col in ("xmin", "ymin", "xmax", "ymax", "width_px", "height_px"):
    ann[col] = pd.to_numeric(ann[col], errors="coerce")
assert inv["sample_id"].is_unique, "inventory sample_id values must be unique"
assert set(ann["sample_id"]) == set(inv["sample_id"]), "annotations and inventory list different images"


def parse_sidecar(path):
    d = json.loads(Path(path).read_text(encoding="utf-8"))
    cam = d.get("camera") or {}
    sensor = (cam.get("sensor") or {}).get("model") or d.get("camera_model") or "unknown"
    roi = ((cam.get("requested") or {}).get("motion") or {}).get("roi")
    ok_roi = (isinstance(roi, dict) and set(roi) >= {"x", "y", "width", "height"}
              and all(isinstance(roi[k], (int, float)) for k in ("x", "y", "width", "height"))
              and roi["width"] > 0 and roi["height"] > 0)
    camera_id = d.get("camera_id") or (d.get("device") or {}).get("camera_id")
    return {
        "camera_id": camera_id or "",
        "sensor": str(sensor),
        "roi": {k: float(roi[k]) for k in ("x", "y", "width", "height")} if ok_roi else None,
        "deployment_id": d.get("deployment_id") or "",
        "motion_selection_source": d.get("motion_selection_source") or "",
        "capture_source": d.get("source") or "",
        "boot_id": (d.get("device") or {}).get("boot_id") or "",
    }


t0 = time.perf_counter()
records, rejected = [], []
boxed_rows = ann[ann["review_state"].eq("boxed") & ann["xmin"].notna()]
boxes_by_sample = {sid: g for sid, g in boxed_rows.groupby("sample_id")}
for row in inv.itertuples(index=False):
    image_path = DATASET_ROOT_PATH / row.image_filename
    sidecar_path = DATASET_ROOT_PATH / row.sidecar_filename
    if not image_path.is_file() or not sidecar_path.is_file():
        rejected.append((row.sample_id, "missing_file")); continue
    data = image_path.read_bytes()
    if hashlib.sha256(data).hexdigest() != row.image_sha256:
        rejected.append((row.sample_id, "image_sha256_mismatch")); continue
    meta = parse_sidecar(sidecar_path)
    g = boxes_by_sample.get(row.sample_id)
    W, H = int(row.width_px), int(row.height_px)
    if g is not None:
        b = g[["xmin", "ymin", "xmax", "ymax"]].to_numpy(np.float32)
        b[:, [0, 2]] = b[:, [0, 2]].clip(0, W); b[:, [1, 3]] = b[:, [1, 3]].clip(0, H)
        keep = (b[:, 2] - b[:, 0] >= 2) & (b[:, 3] - b[:, 1] >= 2)
        boxes, species = b[keep], g["species_name"].fillna("").astype(str).to_numpy()[keep]
    else:
        boxes, species = np.zeros((0, 4), np.float32), np.array([], dtype=object)
    stamp = re.match(r"capture_(\d{8}T\d{6})_(\d{6})Z", row.sample_id)
    when = pd.Timestamp(datetime.strptime("".join(stamp.groups()), "%Y%m%dT%H%M%S%f"), tz="UTC")
    records.append({
        "sample_id": row.sample_id, "jpeg": data, "width": W, "height": H,
        "boxes": boxes, "species": species, "has_bird": row.review_state == "boxed",
        "time": when, **meta,
    })
if rejected:
    print(f"rejected {len(rejected)} images:", pd.Series([r for _, r in rejected]).value_counts().to_dict())
assert records, "no usable images"

frames = pd.DataFrame([{k: v for k, v in r.items() if k not in ("jpeg", "boxes", "species")} for r in records])
frames["n_boxes"] = [len(r["boxes"]) for r in records]
frames["date"] = frames["time"].dt.strftime("%Y-%m-%d")
frames["local_hour"] = frames["time"].dt.tz_convert("America/New_York").dt.hour
print(f"loaded {len(records):,} images ({sum(len(r['jpeg']) for r in records) / 1e6:.0f} MB) in {time.perf_counter() - t0:.0f}s")
display(pd.crosstab([frames["sensor"], frames["camera_id"].replace("", "legacy (no id)")], frames["has_bird"], margins=True))
display(frames["motion_selection_source"].replace("", "(not recorded)").value_counts().to_frame("captures"))


loaded 6,184 images (284 MB) in 269s


has_bird                      False  True   All
sensor camera_id                               
OV2640 esp32cam-389e8af4e9d4    185   429   614
       legacy (no id)           410   596  1006
OV5640 esp32cam-b8aa6d0d5020   1578  2957  4535
       esp32cam-f4ae6d0d5020     20     9    29
All                            2193  3991  6184

,captures
motion_selection_source,
(not recorded),4515
static_roi_fallback,1639
dynamic_motion_blob,30


## 4. Visits, the test split and the folds

Visits come from the hub's own `server/visits.py`, so their IDs match `visits.jsonl`. The test visits are drawn exactly as in v4.1 (same seed and rule), so the test numbers compare directly. Every other visit with a labelled bird is assigned to one of `FOLDS` folds per species, round-robin after a shuffle, so each fold holds a share of every species that has enough visits.


In [4]:
import sys
sys.path.insert(0, str(portable_path(SERVER_DIR)))
from visits import CaptureRef, assign_visits  # the hub's own visit rule

frames["camera_key"] = frames["camera_id"].replace("", "legacy")
visit_of_capture = assign_visits(
    [CaptureRef(sid, cam or None, dep or None, boot or None)
     for sid, cam, dep, boot in zip(frames["sample_id"], frames["camera_id"], frames["deployment_id"], frames["boot_id"])],
    gap_seconds=VISIT_GAP_SECONDS)
frames["visit_id"] = frames["sample_id"].map(visit_of_capture)
visit_of = dict(zip(frames["sample_id"], frames["visit_id"]))

boxes = pd.DataFrame([(i, k, sp, visit_of[r["sample_id"]]) for i, r in enumerate(records)
                      for k, sp in enumerate(r["species"]) if sp], columns=["index", "box", "species", "visit_id"])
visit_species = boxes.groupby("visit_id")["species"].agg(lambda s: s.value_counts().index[0])

# ---- test split: identical to v4.1 (same RNG sequence); v4.1's validation visits join the CV pool ----
rng_split = np.random.default_rng(SEED)
visit_role = {}
for species, visits_of_species in visit_species.groupby(visit_species):
    ids = list(visits_of_species.index)
    rng_split.shuffle(ids)
    n = len(ids)
    n_test = max(int(round(n * TEST_VISIT_FRACTION)), 1) if n >= 3 else 0
    n_val = max(int(round(n * V41_VAL_FRACTION)), 1) if n >= 3 else 0
    for k, v in enumerate(ids):
        visit_role[v] = "test" if k < n_test else "cv"

# ---- folds over the CV pool, per species ----
rng_folds = np.random.default_rng(FOLD_SEED)
visit_fold = {}
pool = visit_species[visit_species.index.map(lambda v: visit_role.get(v) == "cv")]
for species, visits_of_species in pool.groupby(pool):
    ids = sorted(visits_of_species.index)
    rng_folds.shuffle(ids)
    offset = int(rng_folds.integers(FOLDS))  # rotate so small species don't all start in fold 0
    for k, v in enumerate(ids):
        visit_fold[v] = (k + offset) % FOLDS

boxes["split"] = boxes["visit_id"].map(visit_role).fillna("cv")
boxes["fold"] = boxes["visit_id"].map(visit_fold).fillna(-1).astype(int)
frames["split"] = frames["visit_id"].map(visit_role).fillna("cv")

visits = frames.groupby("visit_id").agg(
    camera=("camera_key", "first"), sensor=("sensor", "first"), start=("time", "min"), end=("time", "max"),
    frames=("sample_id", "size"), bird_frames=("has_bird", "sum"), split=("split", "first"))
visits["species"] = visit_species.reindex(visits.index).fillna("")
visits["fold"] = pd.Series(visit_fold).reindex(visits.index)
visits.to_csv(RUN_DIR / "visits.csv")

table = boxes.groupby("species").agg(boxes=("box", "size"), visits=("visit_id", "nunique"))
table["test_visits"] = boxes[boxes["split"].eq("test")].groupby("species")["visit_id"].nunique()
for f in range(FOLDS):
    table[f"fold{f}_visits"] = boxes[boxes["fold"].eq(f)].groupby("species")["visit_id"].nunique()
display(table.fillna(0).astype(int).sort_values("boxes", ascending=False))
pool_boxes = boxes[boxes["split"].eq("cv")]
camera_species = sorted(pool_boxes["species"].unique())
pool_visits_per_species = pool_boxes.groupby("species")["visit_id"].nunique()
print(f"{len(visits):,} visits; CV pool: {pool_boxes['visit_id'].nunique()} visits with birds; "
      f"test: {boxes[boxes['split'].eq('test')]['visit_id'].nunique()} visits")

# check the test visits are the same as in the v4.1 run being compared against
previous_visits = portable_path(PREVIOUS_SPECIES_DIR).parent / "visits.csv"
if previous_visits.is_file():
    prev = pd.read_csv(previous_visits)
    prev_test = set(prev.loc[prev["split"].eq("test"), "visit_id"])
    now_test = set(visits.index[visits["split"].eq("test")]) & set(boxes["visit_id"])
    print("test visits identical to v4.1:", prev_test & set(boxes["visit_id"]) == now_test,
          f"({len(now_test)} visits with birds)")


,boxes,visits,test_visits,fold0_visits,fold1_visits,fold2_visits,fold3_visits
species,,,,,,,
House Sparrow,4529,64,13,14,11,12,14
Mourning Dove,984,11,1,2,2,4,2
Blue Jay,313,19,3,2,5,4,5
American Crow,230,3,1,1,0,0,1
Northern Cardinal,194,15,1,4,4,3,3
Black-capped Chickadee,183,8,0,1,2,3,2
White-breasted Nuthatch,183,10,2,2,1,2,3
Tufted Titmouse,80,3,1,0,1,1,0
Song Sparrow,67,6,1,1,2,1,1


153 visits; CV pool: 75 visits with birds; test: 18 visits
test visits identical to v4.1: True (18 visits with birds)


## 5. Crop geometry

`species_crop_bounds` is the deployed rule: a square crop whose side is `SPECIES_CONTEXT` × the box's long side, centred on the box and shifted inside the frame. Copy this function into the server (see `DEPLOY.md`).


In [5]:
def normalized_roi_bounds(roi, width, height):
    x, y, w, h = (float(roi[k]) for k in ("x", "y", "width", "height"))
    return (max(0, int(math.floor(x * width))), max(0, int(math.floor(y * height))),
            min(width, int(math.ceil((x + w) * width))), min(height, int(math.ceil((y + h) * height))))


def final_crop_bounds(bounds, frame_w, frame_h, margin_fraction=PI_MARGIN_FRACTION, minimum_size=PI_MIN_CROP):
    left, top, right, bottom = bounds
    width, height = right - left, bottom - top
    tw = min(frame_w, max(minimum_size, int(math.ceil(width * (1 + margin_fraction)))))
    th = min(frame_h, max(minimum_size, int(math.ceil(height * (1 + margin_fraction)))))
    cx, cy = (left + right) / 2, (top + bottom) / 2
    cl = min(max(0, int(math.floor(cx - tw / 2))), frame_w - tw)
    ct = min(max(0, int(math.floor(cy - th / 2))), frame_h - th)
    return cl, ct, cl + tw, ct + th


def clip_rect(l, t, r, b, W, H, min_size=8):
    w, h = min(max(r - l, min_size), W), min(max(b - t, min_size), H)
    l = min(max(l, 0), W - w); t = min(max(t, 0), H - h)
    return int(math.floor(l)), int(math.floor(t)), int(math.floor(l)) + int(round(w)), int(math.floor(t)) + int(round(h))


def species_crop_bounds(box, frame_w, frame_h, context=SPECIES_CONTEXT, min_size=SPECIES_MIN_CROP, shift=(0.0, 0.0)):
    """DEPLOYED RULE. box = (left, top, right, bottom) in frame pixels -> half-open integer crop inside the frame."""
    left, top, right, bottom = (float(v) for v in box)
    side = max(right - left, bottom - top) * context
    side = max(side, float(min_size))
    cx = (left + right) / 2 + shift[0] * side
    cy = (top + bottom) / 2 + shift[1] * side
    return clip_rect(cx - side / 2, cy - side / 2, cx + side / 2, cy + side / 2, frame_w, frame_h, min_size)


def visible_fractions(boxes_px, crop):
    if not len(boxes_px):
        return np.zeros(0, np.float32)
    l, t, r, b = crop
    iw = np.clip(np.minimum(boxes_px[:, 2], r) - np.maximum(boxes_px[:, 0], l), 0, None)
    ih = np.clip(np.minimum(boxes_px[:, 3], b) - np.maximum(boxes_px[:, 1], t), 0, None)
    area = np.maximum((boxes_px[:, 2] - boxes_px[:, 0]) * (boxes_px[:, 3] - boxes_px[:, 1]), 1e-6)
    return (iw * ih / area).astype(np.float32)


def iou(a, b):
    iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    return inter / max((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter, 1e-6)


def pi_crop(record):
    """The crop the Pi hands the detector: static ROI (+20%, >=160 px), or the whole frame without a usable ROI."""
    roi = record["roi"]
    if roi is None or roi["width"] >= 0.95:
        return 0, 0, record["width"], record["height"]
    bounds = normalized_roi_bounds(roi, record["width"], record["height"])
    return final_crop_bounds(bounds, record["width"], record["height"])


## 6. Localise the birds: run both detectors on every capture

Two localisers are run on every capture, each on the crop it would see on the hub:

- **`pi_detector`** is the hub's current detector. It runs through the server's own `bird_inference.py` code, on the Pi-selected crop (static ROI with margin).
- **`esp32_roi_v3`** is the v3 ESP32 ROI detector (128 px, int8). It runs on a square around the same crop.

For each one, every labelled box is matched to the detection it overlaps most (IoU ≥ `DETECTOR_MATCH_IOU`). With `LOCALIZER = "auto"`, whichever finds more labelled birds supplies the boxes for the species crops. The table it prints is the answer to "which detector should the hub use to centre the species crop". Results are cached per model hash.


In [6]:
import shutil, sys, types
sys.path.insert(0, str(portable_path(SERVER_DIR)))
import bird_inference as hub  # the Pi's inference module

try:
    from ai_edge_litert.interpreter import Interpreter as _TFLite
except ImportError:
    _TFLite = tf.lite.Interpreter  # the hub's own lookup misses tf.lite in TF >= 2.20


def load_pi_detector():
    path = portable_path(DETECTOR_MODEL)
    try:
        contract = hub.DetectorContract.beside(path)
    except ValueError as error:
        # A Windows git checkout turns the JSON sidecars into CRLF, so their SHA-256 no longer matches the
        # contract. Use LF-normalised copies (the bytes the Pi has); the contract still verifies them.
        if "SHA-256" not in str(error):
            raise
        local = RUN_DIR / "_pi_detector"; local.mkdir(exist_ok=True)
        for f in path.parent.iterdir():
            if f.suffix == ".json":
                (local / f.name).write_bytes(f.read_bytes().replace(b"\r\n", b"\n"))
            elif f == path:
                shutil.copyfile(f, local / f.name)
        path = local / path.name
        contract = hub.DetectorContract.beside(path)
        print("pi_detector JSON files had CRLF line endings; using LF-normalised copies (hashes verified)")
    model = hub.DetectorModel(path, image_size=contract.image_size, num_threads=TFLITE_THREADS, interpreter_factory=_TFLite)
    contract.validate_model(model)
    box_pos, score_pos = int(contract.box_spec["output_position"]), int(contract.score_spec["output_position"])

    def run(rgb, record):
        crop = pi_crop(record)
        l, t, r, b = crop
        out = model.predict(hub.letterbox_array(rgb[t:b, l:r], contract.image_size, padding_value=contract.padding_value))
        return hub.decode_detector_outputs(out[box_pos], out[score_pos], contract, crop)
    return run, sha256_file(path)[:12]


def square_around(crop, W, H):
    l, t, r, b = crop
    side = min(max(r - l, b - t), W, H)
    cx, cy = (l + r) / 2, (t + b) / 2
    L = int(min(max(0, math.floor(cx - side / 2)), W - side)); T = int(min(max(0, math.floor(cy - side / 2)), H - side))
    return L, T, L + side, T + side


def load_roi_v3():
    folder = portable_path(ROI_V3_DIR)
    contract = json.loads((folder / "esp32_detector_contract.json").read_text(encoding="utf-8"))
    path = folder / contract["model_file"]
    interpreter = _TFLite(model_path=str(path), num_threads=TFLITE_THREADS); interpreter.allocate_tensors()
    inp, outs = interpreter.get_input_details()[0], interpreter.get_output_details()
    size = int(contract["input"]["shape"][1])
    anchors = np.asarray(contract["anchors_normalized_wh"], np.float32)
    score_detail = next(o for o in outs if o["shape"][-1] == len(anchors))
    box_detail = next(o for o in outs if o["shape"][-1] == 4 * len(anchors))
    grid = int(box_detail["shape"][1])
    dec = contract["decode"]
    spec = types.SimpleNamespace(box_spec={"shape": [1, grid, grid, len(anchors), 4]}, score_spec={"shape": [1, grid, grid, len(anchors), 1]},
                                 anchors=anchors, candidate_threshold=float(dec["candidate_score_threshold"]), image_size=size,
                                 max_detections=int(dec["max_detections"]), nms_iou=float(dec["nms_iou"]))
    pad = int(contract.get("pi_crop", {}).get("pad_value", 114))

    def dequant(detail):
        raw = interpreter.get_tensor(detail["index"]).astype(np.float32)
        scale, zero = detail["quantization"]
        return (raw - zero) * scale if scale else raw

    def run(rgb, record):
        crop = square_around(pi_crop(record), record["width"], record["height"])
        l, t, r, b = crop
        pixels = hub.letterbox_array(rgb[t:b, l:r], size, padding_value=pad)
        x = np.clip(np.rint(pixels), 0, 255).astype(inp["dtype"])[None] if np.issubdtype(inp["dtype"], np.integer) else pixels[None]
        interpreter.set_tensor(inp["index"], x); interpreter.invoke()
        box = dequant(box_detail).reshape(spec.box_spec["shape"]); score = dequant(score_detail).reshape(spec.score_spec["shape"])
        return hub.decode_detector_outputs(box, score, spec, crop)
    return run, sha256_file(path)[:12]


localizers = {}
for name, loader in (("esp32_roi_v3", load_roi_v3), ("pi_detector", load_pi_detector)):
    try:
        localizers[name] = loader()
    except Exception as error:
        print(f"{name} unavailable:", repr(error))
assert localizers, "no localiser could be loaded"

cache_dir = portable_path(DETECTOR_CACHE_DIR); cache_dir.mkdir(parents=True, exist_ok=True)
detections_by = {}
for name, (run, sha) in localizers.items():
    cache_path = cache_dir / f"{name}_{sha}.json"
    cache = json.loads(cache_path.read_text()) if cache_path.is_file() else {}
    t0, ran, found = time.perf_counter(), 0, []
    for r in records:
        key = r["sample_id"] + "|" + json.dumps(r["roi"], sort_keys=True)
        if key not in cache:
            rgb = np.asarray(Image.open(io.BytesIO(r["jpeg"])).convert("RGB"))
            cache[key] = [[d["score"], d["box"]["left"], d["box"]["top"], d["box"]["right"], d["box"]["bottom"]] for d in run(rgb, r)]
            ran += 1
        found.append([d for d in cache[key] if d[0] >= DETECTOR_MIN_SCORE])
    if ran:
        cache_path.write_text(json.dumps(cache))
    detections_by[name] = found
    print(f"{name}: {ran:,} new captures scored in {time.perf_counter() - t0:.0f}s")


def match_boxes(found):
    matched = {}
    for row in boxes.itertuples(index=False):
        gt = records[row.index]["boxes"][row.box]
        best = max(((iou(gt, d[1:]), d) for d in found[row.index]), default=(0.0, None), key=lambda x: x[0])
        if best[1] is not None and best[0] >= DETECTOR_MATCH_IOU:
            matched[(row.index, row.box)] = (np.asarray(best[1][1:], np.float32), best[1][0], best[0])
    return matched


matches = {name: match_boxes(found) for name, found in detections_by.items()}
boxes["in_pi_crop"] = [visible_fractions(records[i]["boxes"][k:k + 1], pi_crop(records[i]))[0] >= 0.9
                       for i, k in zip(boxes["index"], boxes["box"])]
for name, m in matches.items():
    boxes[f"found_{name}"] = [(i, k) in m for i, k in zip(boxes["index"], boxes["box"])]
    boxes[f"iou_{name}"] = [m.get((i, k), (None, None, 0.0))[2] for i, k in zip(boxes["index"], boxes["box"])]
match_table = boxes.groupby("species")[["in_pi_crop"] + [f"found_{n}" for n in matches]].mean()
match_table.loc["ALL"] = boxes[["in_pi_crop"] + [f"found_{n}" for n in matches]].mean()
display(match_table.round(3))
match_table.to_csv(RUN_DIR / "localisation_by_species.csv")

chosen_localizer = max(matches, key=lambda n: len(matches[n])) if LOCALIZER == "auto" else LOCALIZER
det_box_of = matches[chosen_localizer]
det_sha = localizers[chosen_localizer][1]
boxes["detector_found"] = boxes[f"found_{chosen_localizer}"]
print(f"species crops use: {chosen_localizer} (finds {boxes['detector_found'].mean():.1%} of labelled birds at IoU >= {DETECTOR_MATCH_IOU})")


/home/joe/venvs/ESPclassifier/lib/python3.12/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


pi_detector JSON files had CRLF line endings; using LF-normalised copies (hashes verified)
esp32_roi_v3: 0 new captures scored in 0s
pi_detector: 0 new captures scored in 0s


,in_pi_crop,found_esp32_roi_v3,found_pi_detector
species,,,
American Crow,1.000,0.939,0.017
Black-capped Chickadee,0.989,0.656,0.158
Blue Jay,0.891,0.952,0.160
Downy Woodpecker,0.941,1.000,0.824
House Sparrow,0.960,0.740,0.223
Mourning Dove,0.814,0.971,0.756
Northern Cardinal,0.804,0.943,0.531
Red-bellied Woodpecker,1.000,1.000,0.600
Song Sparrow,1.000,0.657,0.000


species crops use: esp32_roi_v3 (finds 78.9% of labelled birds at IoU >= 0.3)


## 7. Species crops and evaluation tables

Training draws come in three kinds. **detector** is the matched detector box with context and shift jitter. **gt_box** is the hand box, jittered to look like detector error. **loose** puts the bird at 12–45% of the crop side, for when the hub has to fall back to the ROI. Any crop that takes in more than half of a bird of another species is redrawn.

The evaluation kinds (for out-of-fold and test scoring) are `chain` (the detector crop when the detector found the bird, otherwise the Pi's static-ROI crop; this is what the hub would really do), `detector`, `gt_box` and `static_roi`.


In [7]:
pool_boxes = boxes[boxes["split"].eq("cv")]  # re-slice: section 6 added localiser columns

def intruder_free(record, k, crop):
    vis = visible_fractions(record["boxes"], crop)
    other = record["species"] != record["species"][k]
    return not np.any(other & (vis > 0.5))


def jitter_box(rng, box, amount):
    l, t, r, b = (float(v) for v in box)
    w, h = r - l, b - t
    s = math.exp(rng.uniform(-amount, amount))
    cx, cy = (l + r) / 2 + rng.normal(0, amount / 2) * w, (t + b) / 2 + rng.normal(0, amount / 2) * h
    return cx - w * s / 2, cy - h * s / 2, cx + w * s / 2, cy + h * s / 2


def loose_crop(rng, W, H, box):
    l, t, r, b = (float(v) for v in box)
    bw, bh = max(r - l, 2.0), max(b - t, 2.0)
    side = max(bw, bh) / rng.uniform(*LOOSE_BIRD_FRACTION)
    aspect = math.exp(rng.uniform(-0.4, 0.4))
    cw, ch = max(side * math.sqrt(aspect), bw * 1.05), max(side / math.sqrt(aspect), bh * 1.05)
    cx = (l + r) / 2 + rng.uniform(-1, 1) * (cw - bw) / 2 * 0.8
    cy = (t + b) / 2 + rng.uniform(-1, 1) * (ch - bh) / 2 * 0.8
    return clip_rect(cx - cw / 2, cy - ch / 2, cx + cw / 2, cy + ch / 2, W, H, 16)


def train_species_crop(rng, i, k, kind):
    r = records[i]
    W, H = r["width"], r["height"]
    for _ in range(6):
        if kind == "detector" and (i, k) in det_box_of:
            base = det_box_of[(i, k)][0]
        else:
            base = jitter_box(rng, r["boxes"][k], TRAIN_BOX_JITTER) if kind != "loose" else r["boxes"][k]
        if kind == "loose":
            crop = loose_crop(rng, W, H, base)
        else:
            crop = species_crop_bounds(base, W, H, context=rng.uniform(*TRAIN_CONTEXT_RANGE),
                                       shift=tuple(rng.uniform(-TRAIN_SHIFT, TRAIN_SHIFT, 2)))
        if intruder_free(r, k, crop):
            return crop
    return species_crop_bounds(r["boxes"][k], W, H)


def eval_rows(box_subset):
    rows = []
    for row in box_subset.itertuples(index=False):
        r = records[row.index]
        W, H = r["width"], r["height"]
        crops = {"gt_box": species_crop_bounds(r["boxes"][row.box], W, H)}
        if (row.index, row.box) in det_box_of:
            crops["detector"] = species_crop_bounds(det_box_of[(row.index, row.box)][0], W, H)
        pc = pi_crop(r)
        if r["roi"] is not None and row.in_pi_crop and intruder_free(r, row.box, pc):
            crops["static_roi"] = pc
        chain = crops.get("detector", crops.get("static_roi"))
        if chain is not None:
            crops["chain"] = chain
        for kind, crop in crops.items():
            rows.append({"index": row.index, "box": row.box, "sample_id": r["sample_id"], "visit_id": row.visit_id,
                         "species": row.species, "fold": row.fold, "kind": kind,
                         "left": crop[0], "top": crop[1], "right": crop[2], "bottom": crop[3]})
    return pd.DataFrame(rows)


species_test = eval_rows(boxes[boxes["split"].eq("test")])
species_pool = eval_rows(pool_boxes)   # out-of-fold rows are taken from here, fold by fold
display(pd.crosstab(species_test["species"], species_test["kind"]))


kind,chain,detector,gt_box,static_roi
species,,,,
American Crow,78,78,80,0
Blue Jay,26,26,26,21
House Sparrow,277,237,286,270
Mourning Dove,13,13,13,0
Northern Cardinal,2,2,2,2
Song Sparrow,4,4,4,0
Tufted Titmouse,0,0,1,0
White-breasted Nuthatch,8,8,8,6


## 8. Outside crops (iNaturalist + CUB), training only

Only species in the label set are loaded: the ones seen on camera plus `REGIONAL_EXTRA_SPECIES`. Images are EXIF-rotated only when that matches the manifest's recorded size, so the boxes stay aligned. If no image root is found, this section reports it and the species model trains on ESP32 crops only.


In [8]:
def norm_name(name):
    name = re.sub(r"[-_']", " ", str(name).lower())
    return re.sub(r"\s+", " ", name).strip()


CUB_ALIASES = {"cardinal": "northern cardinal", "mockingbird": "northern mockingbird", "tree sparrow": "american tree sparrow"}
camera_species = sorted(boxes["species"].unique())
wanted = {norm_name(s): s for s in camera_species + REGIONAL_EXTRA_SPECIES}
outside = []  # dicts: jpeg, width, height, box, species, source
outside_report = {}


def load_outside_image(path, expected_w, expected_h, box):
    with Image.open(path) as im:
        raw_size = im.size
        rotated = ImageOps.exif_transpose(im)
        if (int(expected_w), int(expected_h)) == rotated.size:
            image = rotated.convert("RGB")
        elif (int(expected_w), int(expected_h)) == raw_size:
            image = im.convert("RGB")
        else:
            return None
    scale = min(1.0, 448 / max(image.size))
    if scale < 1.0:
        image = image.resize((max(1, round(image.width * scale)), max(1, round(image.height * scale))), Image.BILINEAR)
    buffer = io.BytesIO(); image.save(buffer, format="JPEG", quality=92)
    b = np.asarray(box, np.float32) * scale
    b[[0, 2]] = b[[0, 2]].clip(0, image.width); b[[1, 3]] = b[[1, 3]].clip(0, image.height)
    if b[2] - b[0] < 8 or b[3] - b[1] < 8:
        return None
    return buffer.getvalue(), image.width, image.height, b


manifest_dir = next((portable_path(d) for d in OUTSIDE_MANIFEST_DIRS
                     if (portable_path(d) / "bird_manifest.csv").is_file()), None)
inat_root = first_existing(INAT_IMAGE_ROOTS)
cub_root = first_existing(CUB_IMAGE_ROOTS)
print("manifests:", manifest_dir, "| iNat root:", inat_root, "| CUB root:", cub_root)
t0 = time.perf_counter()

if manifest_dir is not None:
    inat = pd.read_csv(manifest_dir / "bird_manifest.csv", encoding="utf-8-sig", low_memory=False,
                       usecols=["image_id", "relative_path", "absolute_path", "width", "height", "common_name"])
    inat["key"] = inat["common_name"].map(norm_name)
    inat = inat[inat["key"].isin(wanted)]
    boxes = pd.read_csv(manifest_dir / "inaturalist_bird_boxes.csv", encoding="utf-8-sig")
    boxes["image_id"] = boxes["source_id"].str.split(":").str[1].astype(int)
    boxes = boxes[boxes["score"] >= OUTSIDE_INAT_MIN_SCORE].sort_values("score", ascending=False).drop_duplicates("image_id")
    inat = inat.merge(boxes[["image_id", "x", "y", "w", "h"]], on="image_id")
    if QUICK_RUN:
        inat = inat.groupby("key").head(3)
    loaded = 0
    for row in inat.itertuples(index=False):
        rel = PurePosixPath(str(row.relative_path).replace("\\", "/"))
        candidates = [portable_path(row.absolute_path), inat_root.joinpath(*rel.parts) if inat_root else None,
                      inat_root.joinpath(*rel.parts[1:]) if inat_root and len(rel.parts) > 1 else None]
        path = next((p for p in candidates if p is not None and p.is_file()), None)
        if path is None:
            continue
        result = load_outside_image(path, row.width, row.height, (row.x, row.y, row.x + row.w, row.y + row.h))
        if result:
            jpeg, w, h, b = result
            outside.append({"jpeg": jpeg, "width": w, "height": h, "box": b, "species": wanted[row.key], "source": "inaturalist"})
            loaded += 1
    outside_report["inaturalist"] = {"candidates": len(inat), "loaded": loaded}

if manifest_dir is not None and (manifest_dir / "cub_200_2011_image_manifest.csv").is_file():
    cub = pd.read_csv(manifest_dir / "cub_200_2011_image_manifest.csv", low_memory=False,
                      usecols=["relative_path", "image_path", "class_name_friendly", "bbox_x", "bbox_y", "bbox_width", "bbox_height",
                               "width_px", "height_px"])
    cub["key"] = cub["class_name_friendly"].map(norm_name).map(lambda k: CUB_ALIASES.get(k, k))
    cub = cub[cub["key"].isin(wanted)]
    if QUICK_RUN:
        cub = cub.groupby("key").head(3)
    loaded = 0
    for row in cub.itertuples(index=False):
        options = [portable_path(row.image_path), cub_root.joinpath(*PurePosixPath(str(row.relative_path).replace("\\", "/")).parts) if cub_root else None]
        path = next((p for p in options if p is not None and p.is_file()), None)
        if path is None:
            continue
        result = load_outside_image(path, row.width_px, row.height_px,
                                    (row.bbox_x, row.bbox_y, row.bbox_x + row.bbox_width, row.bbox_y + row.bbox_height))
        if result:
            jpeg, w, h, b = result
            outside.append({"jpeg": jpeg, "width": w, "height": h, "box": b, "species": wanted[row.key], "source": "cub"})
            loaded += 1
    outside_report["cub"] = {"candidates": len(cub), "loaded": loaded}

print(f"outside crops loaded in {time.perf_counter() - t0:.0f}s:", outside_report)
outside_counts = pd.Series([o["species"] for o in outside], dtype=object).value_counts()
display(outside_counts.to_frame("outside images"))


manifests: /mnt/d/Datasets/ESP32Cam_confirmed/Manifests | iNat root: /mnt/d/Datasets/iNaturalist | CUB root: /mnt/d/Datasets/CUB_200/CUB_200_2011/images
outside crops loaded in 119s: {'inaturalist': {'candidates': 1256, 'loaded': 1256}, 'cub': {'candidates': 1136, 'loaded': 1136}}


,outside images
American Tree Sparrow,107
White-throated Sparrow,107
Purple Finch,106
Chipping Sparrow,105
Northern Mockingbird,105
Carolina Wren,102
Gray Catbird,100
American Goldfinch,100
American Crow,99
Song Sparrow,98


## 9. TensorFlow pipeline, model builder and fixed training schedule

Same crop and augmentation pipeline as v4.1. Training follows one fixed schedule, so the held-out fold never influences the weights.


In [9]:
AUTOTUNE = tf.data.AUTOTUNE


def decode_crop_letterbox(jpeg, crop, size):
    """crop = [left, top, right, bottom] int32, half-open, inside the frame."""
    window = tf.stack([crop[1], crop[0], crop[3] - crop[1], crop[2] - crop[0]])
    image = tf.io.decode_and_crop_jpeg(jpeg, window, channels=3)
    image = tf.image.resize_with_pad(tf.cast(image, tf.float32), size, size, antialias=True)
    return tf.ensure_shape(image, (size, size, 3))


def camera_augment(image, strong_greyscale=0.05):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 18.0)
    image = tf.image.random_contrast(image, 0.85, 1.15)
    image = tf.image.random_saturation(image, 0.8, 1.2)
    image = image * tf.random.uniform([1, 1, 3], 0.92, 1.08)  # white-balance drift between cameras
    grey = tf.image.grayscale_to_rgb(tf.image.rgb_to_grayscale(image))
    image = tf.where(tf.random.uniform([]) < strong_greyscale, grey, image)
    return tf.clip_by_value(image, 0.0, 255.0)


def esp32_look(image, size):
    """Make a sharp iNat/CUB crop look like an ESP32 crop: low native resolution, JPEG, noise, colour cast."""
    native = tf.random.uniform([], 56, 170, dtype=tf.int32)
    image = tf.image.resize(tf.image.resize(image, [native, native], antialias=True), [size, size])
    quality = tf.random.uniform([], 15, 60, dtype=tf.int32)
    image = tf.cast(tf.image.adjust_jpeg_quality(tf.cast(tf.clip_by_value(image, 0, 255), tf.uint8), quality), tf.float32)
    image = image + tf.random.normal(tf.shape(image), 0.0, tf.random.uniform([], 1.0, 8.0))
    image = image * tf.random.uniform([1, 1, 3], 0.8, 1.2)
    return tf.clip_by_value(image, 0.0, 255.0)


def make_train_ds(generator, size, n_classes, onehot):
    signature = (tf.TensorSpec((), tf.string), tf.TensorSpec((4,), tf.int32),
                 tf.TensorSpec((), tf.int32), tf.TensorSpec((), tf.bool))

    def prepare(jpeg, crop, label, outside):
        image = decode_crop_letterbox(jpeg, crop, size)
        image = tf.cond(outside, lambda: esp32_look(image, size), lambda: image)
        image = camera_augment(image)
        target = tf.one_hot(label, n_classes) if onehot else tf.cast(tf.reshape(label, (1,)), tf.float32)
        return image, target

    return (tf.data.Dataset.from_generator(generator, output_signature=signature)
            .map(prepare, num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE, drop_remainder=True).prefetch(AUTOTUNE))


def make_eval_ds(jpegs, crops, size):
    ds = tf.data.Dataset.from_tensor_slices((list(jpegs), np.asarray(crops, np.int32)))
    return ds.map(lambda j, c: decode_crop_letterbox(j, c, size), num_parallel_calls=AUTOTUNE).batch(64).prefetch(AUTOTUNE)


def build_classifier(size, alpha, n_out, activation, name):
    backbone = keras.applications.MobileNetV2(input_shape=(size, size, 3), alpha=alpha, include_top=False, weights=WEIGHTS)
    backbone.trainable = False
    inputs = keras.Input((size, size, 3), name="image")
    x = keras.layers.Rescaling(1.0 / 127.5, offset=-1.0, name="scale_to_mobilenet")(inputs)
    x = backbone(x, training=False)  # BatchNorm stays in inference mode, even while fine-tuning
    x = keras.layers.GlobalAveragePooling2D(name="global_average_pool")(x)
    x = keras.layers.Dropout(0.3, name="dropout")(x)
    outputs = keras.layers.Dense(n_out, activation=activation, name="output")(x)
    return keras.Model(inputs, outputs, name=name), backbone


def unfreeze_from(backbone, layer_name):
    backbone.trainable = True
    names = [layer.name for layer in backbone.layers]
    start = names.index(layer_name)
    for k, layer in enumerate(backbone.layers):
        layer.trainable = k >= start and not isinstance(layer, keras.layers.BatchNormalization)
    return sum(1 for layer in backbone.layers if layer.trainable)


try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    Interpreter = tf.lite.Interpreter


def export_tflite(model, representative_images, out_dir, stem="model"):
    out_dir = Path(out_dir)
    float_path, uint8_path = out_dir / f"{stem}_float32.tflite", out_dir / f"{stem}_uint8.tflite"
    float_path.write_bytes(tf.lite.TFLiteConverter.from_keras_model(model).convert())
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = lambda: ([img[None].astype(np.float32)] for img in representative_images)
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.uint8
    converter.inference_output_type = tf.uint8
    uint8_path.write_bytes(converter.convert())
    model.save(out_dir / f"{stem}.keras")
    return float_path, uint8_path


def tflite_run(path, images_ds):
    """-> raw outputs (N, K), dequantized outputs (N, K), tensor info."""
    interpreter = Interpreter(model_path=str(path), num_threads=TFLITE_THREADS)
    interpreter.allocate_tensors()
    inp, out = interpreter.get_input_details()[0], interpreter.get_output_details()[0]
    raw_rows = []
    for batch in images_ds:
        for image in batch.numpy():
            x = image[None].astype(np.float32)
            if np.issubdtype(inp["dtype"], np.integer):
                scale, zero = inp["quantization"]
                x = np.clip(np.rint(x / scale + zero), 0, 255).astype(inp["dtype"])
            interpreter.set_tensor(inp["index"], x)
            interpreter.invoke()
            raw_rows.append(interpreter.get_tensor(out["index"]).reshape(-1).copy())
    raw = np.stack(raw_rows)
    if np.issubdtype(out["dtype"], np.integer):
        scale, zero = out["quantization"]
        values = (raw.astype(np.float32) - zero) * scale
    else:
        values = raw.astype(np.float32)
    info = {"input_shape": [int(v) for v in inp["shape"]], "input_dtype": np.dtype(inp["dtype"]).name,
            "input_quantization": [float(inp["quantization"][0]), int(inp["quantization"][1])],
            "output_dtype": np.dtype(out["dtype"]).name,
            "output_quantization": [float(out["quantization"][0]), int(out["quantization"][1])]}
    return raw, values, info


def train_fixed(model, backbone, train_ds, loss, out_dir, head_epochs=HEAD_EPOCHS, fine_tune_epochs=FINE_TUNE_EPOCHS):
    """Fixed schedule, no early stopping: head only, then fine-tune with warm-up + cosine decay to ~0."""
    steps = max(1, EXAMPLES_PER_EPOCH // BATCH_SIZE)
    model.compile(optimizer=keras.optimizers.Adam(HEAD_LR), loss=loss)
    h1 = model.fit(train_ds, epochs=head_epochs, steps_per_epoch=steps, verbose=2)
    unfreeze_from(backbone, FINE_TUNE_FROM_BLOCK)
    total = steps * fine_tune_epochs
    schedule = keras.optimizers.schedules.CosineDecay(
        FINE_TUNE_PEAK_LR * 0.05, decay_steps=max(1, total - steps), alpha=0.01,
        warmup_target=FINE_TUNE_PEAK_LR, warmup_steps=steps)
    model.compile(optimizer=keras.optimizers.AdamW(schedule, weight_decay=WEIGHT_DECAY), loss=loss)
    h2 = model.fit(train_ds, epochs=fine_tune_epochs, steps_per_epoch=steps, verbose=2)
    history = pd.DataFrame({"epoch": range(1, head_epochs + fine_tune_epochs + 1),
                            "stage": ["head"] * head_epochs + ["fine_tune"] * fine_tune_epochs,
                            "loss": h1.history["loss"] + h2.history["loss"]})
    history.to_csv(Path(out_dir) / "training_history.csv", index=False)
    return history


## 10. Cross-validation: train each arm once per fold, predict the fold it didn't see

Every fold model trains on the other folds' visits (plus outside crops for `esp32_plus_outside`) and scores every crop of its own fold with the uint8 TFLite model, which is what the hub runs. Each arm uses one fixed class list across all folds, so the out-of-fold probabilities line up.


In [10]:
def arm_classes(arm):
    if arm == "esp32_only" or not outside:
        return list(camera_species)
    return sorted(set(camera_species) | set(outside_counts.index))


def species_generator_factory(classes, train_subset, use_outside, seed):
    index_of = {c: k for k, c in enumerate(classes)}
    by_class = {c: [v[["index", "box"]].to_numpy() for _, v in g.groupby("visit_id")]
                for c, g in train_subset.groupby("species") if c in index_of}
    esp_classes = sorted(by_class)
    esp_w = np.array([len(by_class[c]) ** SPECIES_CLASS_BALANCE_POWER for c in esp_classes], np.float64)
    esp_w /= esp_w.sum()
    out_by_class = {}
    if use_outside:
        for k, o in enumerate(outside):
            if o["species"] in index_of:
                out_by_class.setdefault(o["species"], []).append(k)
    out_classes = sorted(out_by_class)
    out_w = np.array([len(out_by_class[c]) ** SPECIES_CLASS_BALANCE_POWER for c in out_classes], np.float64) if out_classes else None
    if out_w is not None:
        out_w /= out_w.sum()
    esp_share = SPECIES_ESP32_SHARE if out_classes else 1.0
    kinds, kind_w = list(CROP_MIX), np.array(list(CROP_MIX.values()), np.float64) / sum(CROP_MIX.values())

    def generator():
        rng = np.random.default_rng(seed)
        while True:
            if rng.random() < esp_share:
                c = esp_classes[rng.choice(len(esp_classes), p=esp_w)]
                visit = by_class[c][rng.integers(len(by_class[c]))]
                i, k = visit[rng.integers(len(visit))]
                kind = kinds[rng.choice(len(kinds), p=kind_w)]
                yield records[i]["jpeg"], np.asarray(train_species_crop(rng, int(i), int(k), kind), np.int32), np.int32(index_of[c]), False
            else:
                c = out_classes[rng.choice(len(out_classes), p=out_w)]
                o = outside[out_by_class[c][rng.integers(len(out_by_class[c]))]]
                if rng.random() < 0.8:
                    crop = species_crop_bounds(jitter_box(rng, o["box"], TRAIN_BOX_JITTER), o["width"], o["height"],
                                               context=rng.uniform(*TRAIN_CONTEXT_RANGE))
                else:
                    crop = loose_crop(rng, o["width"], o["height"], o["box"])
                yield o["jpeg"], np.asarray(crop, np.int32), np.int32(index_of[c]), True
    return generator


def score_table(path, table):
    ds = make_eval_ds([records[i]["jpeg"] for i in table["index"]],
                      table[["left", "top", "right", "bottom"]].to_numpy(np.int32), SPECIES_IMG)
    return tflite_run(path, ds)


def train_export(arm, classes, train_subset, out_dir, seed):
    out_dir.mkdir(parents=True, exist_ok=True)
    gen = species_generator_factory(classes, train_subset, use_outside=(arm != "esp32_only"), seed=seed)
    ds = make_train_ds(gen, SPECIES_IMG, len(classes), onehot=True)
    model, backbone = build_classifier(SPECIES_IMG, SPECIES_ALPHA, len(classes), "softmax", "species_mobilenetv2")
    train_fixed(model, backbone, ds, keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING), out_dir)
    (out_dir / "labels.txt").write_text("\n".join(classes) + "\n", encoding="utf-8")
    rep = []
    for jpeg, crop, _, is_out in gen():
        image = decode_crop_letterbox(jpeg, tf.constant(crop), SPECIES_IMG)
        rep.append((esp32_look(image, SPECIES_IMG) if is_out else image).numpy())
        if len(rep) >= REPRESENTATIVE_SAMPLES:
            break
    float_path, uint8_path = export_tflite(model, rep, out_dir)
    del model, backbone, ds
    keras.backend.clear_session()
    return float_path, uint8_path


arms = [a for a in SPECIES_ARMS if not (a == "esp32_plus_outside" and not outside)]
oof = {}  # arm -> (table rows of the pool, probabilities aligned to those rows)
cv_minutes = {}
for arm in arms:
    classes = arm_classes(arm)
    probs = np.full((len(species_pool), len(classes)), np.nan, np.float32)
    t0 = time.perf_counter()
    for fold in range(FOLDS):
        print(f"\n=== {arm}: fold {fold + 1}/{FOLDS} ({len(classes)} classes) ===")
        train_subset = pool_boxes[pool_boxes["fold"].ne(fold)]
        _, uint8_path = train_export(arm, classes, train_subset, RUN_DIR / "cv" / arm / f"fold{fold}", SEED + 100 * fold)
        mask = species_pool["fold"].eq(fold).to_numpy()
        if mask.any():
            _, p, _ = score_table(uint8_path, species_pool[mask])
            probs[mask] = p
    assert not np.isnan(probs).any(), "every pool crop must get an out-of-fold prediction"
    oof[arm] = (classes, probs)
    cv_minutes[arm] = (time.perf_counter() - t0) / 60
    np.save(RUN_DIR / "cv" / arm / "oof_probabilities.npy", probs)
    print(f"{arm}: {FOLDS} folds in {cv_minutes[arm]:.1f} min")
species_pool.to_csv(RUN_DIR / "cv" / "oof_rows.csv", index=False)



=== esp32_plus_outside: fold 1/4 (31 classes) ===


I0000 00:00:1795107878.334924   22126 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3582 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Epoch 1/3


/home/joe/venvs/ESPclassifier/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1795107880.165347   22126 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1795107882.603738   22291 service.cc:153] XLA service 0x7b5808032b30 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1795107882.603820   22291 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3060 Laptop GPU, Compute Capability 8.6 (Driver: 13.4.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.27.0)
I0000 00:00:1795107882.730304   22291 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1

300/300 - 19s - 65ms/step - loss: 2.2951
Epoch 2/3
300/300 - 9s - 31ms/step - loss: 1.7331
Epoch 3/3
300/300 - 10s - 32ms/step - loss: 1.5833
Epoch 1/15


I0000 00:00:1795107926.062523   22290 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_24882__.191


... (115 outputs left out)

INFO:tensorflow:Assets written to: /tmp/tmpdez7fgmz/assets


Saved artifact at '/tmp/tmpdez7fgmz'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='image')
Output Type:
  TensorSpec(shape=(None, 11), dtype=tf.float32, name=None)
Captures:
  135620179056336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135620179057296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135620179057680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135620179054992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135620179058448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135620179055952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135620179054608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135620179053264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135620179054800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135620179054416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135620179053072: Ten

/home/joe/venvs/ESPclassifier/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1795110292.388418   22126 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1795110292.388471   22126 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1795110292.388626   22126 reader.cc:83] Reading SavedModel from: /tmp/tmpdez7fgmz
I0000 00:00:1795110292.399926   22126 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1795110292.399959   22126 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpdez7fgmz
I0000 00:00:1795110292.506749   22126 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1795110293.022605   22126 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpdez7fgmz
I0000 00:00:1795110293.139863   22126 loader.cc:471] SavedModel load for ta

esp32_only: 4 folds in 20.3 min


## 11. Out-of-fold evaluation, auto-label thresholds and the choice of arm

All of this uses the out-of-fold predictions, so every non-test visit counts as validation.

- **Species threshold:** the lowest mean probability at which at least `MIN_AUTO_TRACKS` tracks are auto-labelled at `AUTO_TARGET_PRECISION` or better. Only species with at least `MIN_TRAIN_VISITS_FOR_AUTO` visits on at least `MIN_DAYS_FOR_AUTO` different days can be auto-labelled; a species seen on a single day mostly teaches the model that day's scene.
- **Group threshold:** the same rule, applied only to tracks that fall below the species threshold.
- **Arm:** the one that auto-labels the largest share of tracks; out-of-fold visit macro recall breaks ties.

Wilson 95% lower bounds show how much the precision could really be with this many tracks.


In [11]:
group_of = {s: g for g, members in SPECIES_GROUPS.items() for s in members}
pool_days_per_species = pool_boxes.assign(day=frames.loc[pool_boxes["index"], "date"].to_numpy()).groupby("species")["day"].nunique()
allowed_auto = {s for s, n in pool_visits_per_species.items()
                if n >= MIN_TRAIN_VISITS_FOR_AUTO and pool_days_per_species.get(s, 0) >= MIN_DAYS_FOR_AUTO}
print("species eligible for auto-labels:", sorted(allowed_auto))
print("held back (too few visits or a single day):", sorted(set(pool_visits_per_species.index) - allowed_auto))


def wilson_low(successes, n, z=1.96):
    if n == 0:
        return float("nan")
    p = successes / n
    return float((p + z * z / (2 * n) - z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))) / (1 + z * z / n))


def frame_metrics(table, probs, classes):
    idx = {c: k for k, c in enumerate(classes)}
    truth = table["species"].map(idx).fillna(-1).to_numpy(int)
    pred = probs.argmax(axis=1)
    known = truth >= 0
    recalls = {classes[c]: float(np.mean(pred[truth == c] == c)) for c in np.unique(truth[known])}
    return {"crops": int(known.sum()), "top1": float(np.mean(pred[known] == truth[known])) if known.any() else np.nan,
            "macro_recall": float(np.mean(list(recalls.values()))) if recalls else np.nan}, recalls


def track_probs(table, probs):
    mask = table["kind"].eq("chain").to_numpy()
    sub = table[mask].reset_index(drop=True)
    p = probs[mask]
    rows, mats = [], []
    for (visit, species), g in sub.groupby(["visit_id", "species"]):
        rows.append({"visit_id": visit, "species": species, "frames": len(g)})
        mats.append(p[g.index.to_numpy()].mean(axis=0))
    return pd.DataFrame(rows), (np.stack(mats) if mats else np.zeros((0, probs.shape[1])))


def group_probs(p, classes):
    groups = sorted(set(group_of.get(c, c) for c in classes))
    gp = np.zeros((len(p), len(groups)))
    for k, c in enumerate(classes):
        gp[:, groups.index(group_of.get(c, c))] += p[:, k]
    return gp, groups


def choose_threshold(conf, correct, eligible):
    best = 1.01
    for t in sorted(set(np.round(conf[eligible], 3)), reverse=True):
        keep = eligible & (conf >= t)
        if keep.sum() < MIN_AUTO_TRACKS:
            continue
        if correct[keep].mean() >= AUTO_TARGET_PRECISION:
            best = float(t)
        else:
            break
    return best


def decisions(tracks, p, classes, t_species, t_group):
    pred = np.array([classes[k] for k in p.argmax(axis=1)]); conf = p.max(axis=1)
    gp, groups = group_probs(p, classes)
    gpred = np.array([groups[k] for k in gp.argmax(axis=1)]); gconf = gp.max(axis=1)
    species_auto = np.isin(pred, list(allowed_auto)) & (conf >= t_species)
    group_auto = ~species_auto & (gconf >= t_group)
    truth = tracks["species"].to_numpy()
    out = tracks.assign(predicted=pred, confidence=conf, group_predicted=gpred, group_confidence=gconf,
                        decision=np.where(species_auto, "species", np.where(group_auto, "group", "review")))
    out["correct"] = np.where(species_auto, pred == truth,
                              np.where(group_auto, gpred == np.array([group_of.get(s, s) for s in truth]), np.nan))
    return out


def automation_rows(decided, arm, split):
    rows = []
    for decision in ("species", "group", "review"):
        part = decided[decided["decision"].eq(decision)]
        n, ok = len(part), int(np.nansum(part["correct"])) if decision != "review" else 0
        rows.append({"arm": arm, "split": split, "decision": decision, "tracks": n, "share": n / max(len(decided), 1),
                     "precision": ok / n if n and decision != "review" else np.nan,
                     "precision_wilson95_low": wilson_low(ok, n) if decision != "review" else np.nan})
    return rows


cv_rows, cv_automation, cv_info = [], [], {}
for arm, (classes, probs) in oof.items():
    for kind in ("chain", "detector", "gt_box", "static_roi"):
        mask = species_pool["kind"].eq(kind).to_numpy()
        if mask.any():
            m, _ = frame_metrics(species_pool[mask], probs[mask], classes)
            cv_rows.append({"arm": arm, "unit": f"frame:{kind}", **m})
    tracks, tp = track_probs(species_pool, probs)
    idx = {c: k for k, c in enumerate(classes)}
    truth = tracks["species"].map(idx).fillna(-1).to_numpy(int)
    hit = tp.argmax(axis=1) == truth
    per_species = {c: float(hit[truth == idx[c]].mean()) for c in tracks["species"].unique() if c in idx}
    cv_rows.append({"arm": arm, "unit": "visit_track", "crops": len(tracks), "top1": float(hit.mean()),
                    "macro_recall": float(np.mean(list(per_species.values())))})
    pred = np.array([classes[k] for k in tp.argmax(axis=1)])
    t_species = choose_threshold(tp.max(axis=1), pred == tracks["species"].to_numpy(), np.isin(pred, list(allowed_auto)))
    species_auto = np.isin(pred, list(allowed_auto)) & (tp.max(axis=1) >= t_species)
    gp, groups = group_probs(tp, classes)
    gpred = np.array([groups[k] for k in gp.argmax(axis=1)])
    gtruth = np.array([group_of.get(s, s) for s in tracks["species"]])
    t_group = choose_threshold(gp.max(axis=1), gpred == gtruth, ~species_auto)
    decided = decisions(tracks, tp, classes, t_species, t_group)
    cv_automation += automation_rows(decided, arm, "out_of_fold")
    cv_info[arm] = {"classes": classes, "t_species": t_species, "t_group": t_group, "tracks": tracks, "track_probs": tp,
                    "decided": decided, "auto_share": float((decided["decision"] != "review").mean()),
                    "visit_macro_recall": cv_rows[-1]["macro_recall"], "per_species_visit_recall": per_species}
    decided.to_csv(RUN_DIR / "cv" / arm / "oof_track_decisions.csv", index=False)

cv_results = pd.DataFrame(cv_rows); cv_results.to_csv(RUN_DIR / "cv_results.csv", index=False)
cv_automation = pd.DataFrame(cv_automation); cv_automation.to_csv(RUN_DIR / "cv_automation.csv", index=False)
display(cv_results.round(3)); display(cv_automation.round(3))
best_arm = max(cv_info, key=lambda a: (cv_info[a]["auto_share"], cv_info[a]["visit_macro_recall"]))
for arm, info in cv_info.items():
    fmt_t = lambda t: "never" if t > 1 else f"{t:.3f}"
    print(f"{arm}: auto-labels {info['auto_share']:.1%} of {len(info['tracks'])} out-of-fold tracks "
          f"(t_species {fmt_t(info['t_species'])}, t_group {fmt_t(info['t_group'])}), visit macro recall {info['visit_macro_recall']:.3f}")
print("selected arm:", best_arm)
display(pd.Series(cv_info[best_arm]["per_species_visit_recall"], name="out-of-fold visit recall").to_frame()
        .join(pool_visits_per_species.rename("pool visits")).round(3))


,arm,unit,crops,top1,macro_recall
0,esp32_plus_outside,frame:chain,5867,0.658,0.492
1,esp32_plus_outside,frame:detector,4987,0.631,0.497
2,esp32_plus_outside,frame:gt_box,6365,0.608,0.513
3,esp32_plus_outside,frame:static_roi,4070,0.729,0.238
4,esp32_plus_outside,visit_track,109,0.761,0.672
5,esp32_only,frame:chain,5867,0.769,0.438
6,esp32_only,frame:detector,4987,0.752,0.442
7,esp32_only,frame:gt_box,6365,0.748,0.416
8,esp32_only,frame:static_roi,4070,0.796,0.214
9,esp32_only,visit_track,109,0.661,0.441


,arm,split,decision,tracks,share,precision,precision_wilson95_low
0,esp32_plus_outside,out_of_fold,species,14,0.128,1.0,0.785
1,esp32_plus_outside,out_of_fold,group,0,0.000,NaN,NaN
2,esp32_plus_outside,out_of_fold,review,95,0.872,NaN,NaN
3,esp32_only,out_of_fold,species,0,0.000,NaN,NaN
4,esp32_only,out_of_fold,group,0,0.000,NaN,NaN
5,esp32_only,out_of_fold,review,109,1.000,NaN,NaN


esp32_plus_outside: auto-labels 12.8% of 109 out-of-fold tracks (t_species 0.770, t_group never), visit macro recall 0.672
esp32_only: auto-labels 0.0% of 109 out-of-fold tracks (t_species never, t_group never), visit macro recall 0.441
selected arm: esp32_plus_outside


,out-of-fold visit recall,pool visits
House Sparrow,0.932,51
Downy Woodpecker,0.500,4
Mourning Dove,0.800,10
Northern Cardinal,0.462,14
Blue Jay,0.800,16
Black-capped Chickadee,0.400,8
White-breasted Nuthatch,0.500,8
Tufted Titmouse,0.000,2
Red-bellied Woodpecker,1.000,1
Song Sparrow,1.000,5


## 12. Final model: retrain on every non-test visit, score test once, export

The chosen arm is retrained on the whole CV pool with the same fixed schedule. It keeps the thresholds chosen out-of-fold and is scored once on the test visits. That's the same test as v4.1, so the comparison table is like for like.

`review_queue.csv` uses the **out-of-fold** predictions: each training visit is judged by a model that never saw it, so confident disagreements are real label-error candidates.


In [12]:
import shutil
classes = cv_info[best_arm]["classes"]
t_species, t_group = cv_info[best_arm]["t_species"], cv_info[best_arm]["t_group"]
t0 = time.perf_counter()
final_dir = RUN_DIR / "final"
float_path, uint8_path = train_export(best_arm, classes, pool_boxes, final_dir, SEED)
final_minutes = (time.perf_counter() - t0) / 60
for src, name in ((uint8_path, "model_uint8.tflite"), (float_path, "model_float32.tflite"), (final_dir / "labels.txt", "labels.txt")):
    shutil.copyfile(src, RUN_DIR / name)

_, test_probs, tensor_info = score_table(RUN_DIR / "model_uint8.tflite", species_test)
test_rows = []
for kind in ("chain", "detector", "gt_box", "static_roi"):
    mask = species_test["kind"].eq(kind).to_numpy()
    if mask.any():
        m, _ = frame_metrics(species_test[mask], test_probs[mask], classes)
        test_rows.append({"model": f"v4.1-CV ({best_arm})", "unit": f"frame:{kind}", **m})
test_tracks, test_tp = track_probs(species_test, test_probs)
idx = {c: k for k, c in enumerate(classes)}
truth = test_tracks["species"].map(idx).fillna(-1).to_numpy(int)
hit = test_tp.argmax(axis=1) == truth
test_rows.append({"model": f"v4.1-CV ({best_arm})", "unit": "visit_track", "crops": len(test_tracks), "top1": float(hit.mean()),
                  "macro_recall": float(np.mean([hit[truth == c].mean() for c in np.unique(truth[truth >= 0])]))})
test_decided = decisions(test_tracks, test_tp, classes, t_species, t_group)
test_automation = pd.DataFrame(automation_rows(test_decided, best_arm, "test"))
test_decided.to_csv(RUN_DIR / "test_track_decisions.csv", index=False)

# v4.1 (camera-only arm, fixed validation split) on exactly the same test crops
try:
    prev_dir = portable_path(PREVIOUS_SPECIES_DIR)
    prev_classes = [l.strip() for l in (prev_dir / "labels.txt").read_text(encoding="utf-8").splitlines() if l.strip()]
    _, prev_probs, _ = score_table(prev_dir / "model_uint8.tflite", species_test)
    for kind in ("chain", "static_roi"):
        mask = species_test["kind"].eq(kind).to_numpy()
        m, _ = frame_metrics(species_test[mask], prev_probs[mask], prev_classes)
        test_rows.append({"model": "v4.1 (esp32_only)", "unit": f"frame:{kind}", **m})
    pt, ptp = track_probs(species_test, prev_probs)
    pidx = {c: k for k, c in enumerate(prev_classes)}
    ptruth = pt["species"].map(pidx).fillna(-1).to_numpy(int)
    phit = ptp.argmax(axis=1) == ptruth
    test_rows.append({"model": "v4.1 (esp32_only)", "unit": "visit_track", "crops": len(pt), "top1": float(phit.mean()),
                      "macro_recall": float(np.mean([phit[ptruth == c].mean() for c in np.unique(ptruth[ptruth >= 0])]))})
except Exception as error:
    print("v4.1 comparison skipped:", repr(error))
test_results = pd.DataFrame(test_rows); test_results.to_csv(RUN_DIR / "test_results.csv", index=False)
test_automation.to_csv(RUN_DIR / "test_automation.csv", index=False)
display(test_results.round(3)); display(test_automation.round(3))

# confusion on test chain crops
mask = species_test["kind"].eq("chain").to_numpy() & species_test["species"].isin(classes).to_numpy()
if mask.any():
    matrix = pd.crosstab(pd.Series(species_test["species"][mask].to_numpy(), name="true"),
                         pd.Series([classes[k] for k in test_probs[mask].argmax(axis=1)], name="predicted"))
    matrix.to_csv(RUN_DIR / "test_confusion_chain.csv"); display(matrix)

# review queue from out-of-fold predictions (+ test)
oof_tracks = cv_info[best_arm]["decided"].assign(split="cv")
queue = pd.concat([oof_tracks, test_decided.assign(split="test")], ignore_index=True)
queue["label_prob"] = [cv_info[best_arm]["track_probs"][n, idx[s]] if (sp == "cv" and s in idx) else np.nan
                       for n, (s, sp) in enumerate(zip(queue["species"], queue["split"]))]
queue = queue[(queue["predicted"] != queue["species"]) & (queue["confidence"] >= REVIEW_DISAGREEMENT_MIN_PROB)]
examples = pd.concat([species_pool, species_test]).query("kind == 'chain'").groupby(["visit_id", "species"])["sample_id"].agg(lambda s: " ".join(list(s)[:3]))
queue["example_frames"] = [examples.get((v, s), "") for v, s in zip(queue["visit_id"], queue["species"])]
queue = queue.sort_values("confidence", ascending=False)
queue.to_csv(RUN_DIR / "review_queue.csv", index=False)
print(f"review queue: {len(queue)} tracks where a model that never saw the visit confidently disagrees with its label")
display(queue.head(20))

species_contract = {
    "model_file": "model_uint8.tflite", "float_model_file": "model_float32.tflite", "labels_file": "labels.txt",
    "task": "species of one localised bird", "arm": best_arm, "classes": classes,
    "camera_trained_classes": camera_species, "outside_only_classes": sorted(set(classes) - set(camera_species)),
    "input_shape": tensor_info["input_shape"], "input_dtype": tensor_info["input_dtype"],
    "input_quantization": tensor_info["input_quantization"],
    "output_dtype": tensor_info["output_dtype"], "output_quantization": tensor_info["output_quantization"],
    "output": "softmax probabilities in labels.txt order",
    "geometry": "detector_box_square_context_crop_then_letterbox", "geometry_location": "outside the model",
    "box_source": "localiser / detector box in original frame pixels (fallback: the isBird selected crop)",
    "crop_rule": {"context": SPECIES_CONTEXT, "min_size_px": SPECIES_MIN_CROP, "shape": "square, centred, shifted inside the frame"},
    "color_order": "RGB", "jpeg_decode_channels": 3, "resize_method": "TensorFlow bilinear", "resize_antialias": True,
    "letterbox_padding_value": 0, "numeric_scaling": "[0,255] to [-1,1] inside the model",
    "visit_aggregation": {"gap_seconds": VISIT_GAP_SECONDS, "rule": "mean probability of the best-localised bird per capture"},
    "automation": {"species_min_confidence": t_species if t_species <= 1 else None,
                   "group_min_confidence": t_group if t_group <= 1 else None,
                   "auto_species": sorted(allowed_auto & set(classes)), "groups": SPECIES_GROUPS,
                   "target_precision": AUTO_TARGET_PRECISION, "selected_on": f"{FOLDS}-fold out-of-fold visit tracks"},
    "localizer": chosen_localizer, "localizer_model_sha256_prefix": det_sha,
}
write_json(RUN_DIR / "species_contract.json", species_contract)
print(f"final model trained in {final_minutes:.1f} min; files in {RUN_DIR}")


Epoch 1/3


/home/joe/venvs/ESPclassifier/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1795110334.363189   22290 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_984875__.117


300/300 - 17s - 56ms/step - loss: 2.2786
Epoch 2/3
300/300 - 11s - 37ms/step - loss: 1.7658
Epoch 3/3
300/300 - 11s - 37ms/step - loss: 1.5972
Epoch 1/15


I0000 00:00:1795110378.110160   22289 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1001666__.191


300/300 - 23s - 78ms/step - loss: 1.4930
Epoch 2/15
300/300 - 12s - 40ms/step - loss: 1.3038
Epoch 3/15
300/300 - 12s - 41ms/step - loss: 1.0584
Epoch 4/15
300/300 - 12s - 41ms/step - loss: 0.9715
Epoch 5/15
300/300 - 12s - 41ms/step - loss: 0.8749
Epoch 6/15
300/300 - 12s - 41ms/step - loss: 0.8247
Epoch 7/15
300/300 - 13s - 43ms/step - loss: 0.7770
Epoch 8/15
300/300 - 18s - 59ms/step - loss: 0.7070
Epoch 9/15
300/300 - 17s - 58ms/step - loss: 0.6755
Epoch 10/15
300/300 - 18s - 59ms/step - loss: 0.6180
Epoch 11/15
300/300 - 17s - 56ms/step - loss: 0.5901
Epoch 12/15
300/300 - 16s - 54ms/step - loss: 0.5622
Epoch 13/15
300/300 - 15s - 49ms/step - loss: 0.5462
Epoch 14/15
300/300 - 17s - 55ms/step - loss: 0.5339
Epoch 15/15
300/300 - 16s - 55ms/step - loss: 0.5320
INFO:tensorflow:Assets written to: /tmp/tmp8dx80s5q/assets


INFO:tensorflow:Assets written to: /tmp/tmp8dx80s5q/assets


... (8 outputs left out)

predicted,American Crow,Black-capped Chickadee,Blue Jay,Downy Woodpecker,House Sparrow,Mourning Dove,Northern Cardinal,Song Sparrow,Tufted Titmouse,White-breasted Nuthatch
true,,,,,,,,,,
American Crow,77,0,0,0,0,0,0,0,0,1
Blue Jay,1,0,6,8,10,0,0,0,1,0
House Sparrow,9,1,0,1,236,1,0,3,1,25
Mourning Dove,1,0,1,0,2,0,0,8,0,1
Northern Cardinal,0,0,0,0,0,0,2,0,0,0
Song Sparrow,0,0,0,0,0,0,0,4,0,0
White-breasted Nuthatch,0,0,0,0,0,0,0,2,0,6


review queue: 14 tracks where a model that never saw the visit confidently disagrees with its label


,visit_id,species,frames,predicted,confidence,group_predicted,group_confidence,decision,correct,split,label_prob,example_frames
126,legacy/capture_20261007T164032_688006Z_b30dd1dd,White-breasted Nuthatch,2,Song Sparrow,0.859375,sparrow-like,0.888672,species,0.0,test,NaN,capture_20261007T164032_688006Z_b30dd1dd captu...
125,legacy/capture_20261007T164032_688006Z_b30dd1dd,House Sparrow,2,Song Sparrow,0.837891,sparrow-like,0.896484,species,0.0,test,NaN,capture_20261007T164243_163703Z_108bac7b captu...
52,esp32cam-b8aa6d0d5020/capture_20261116T212636_...,Northern Cardinal,2,House Sparrow,0.769531,sparrow-like,0.824219,review,NaN,cv,0.017578,capture_20261116T212636_448167Z_bc5c6815 captu...
37,esp32cam-b8aa6d0d5020/capture_20261028T201614_...,Downy Woodpecker,1,Blue Jay,0.687500,jay/crow,0.738281,review,NaN,cv,0.003906,capture_20261028T201614_652204Z_a0f47211
105,legacy/capture_20261014T215358_419298Z_e05bba42,Mourning Dove,3,Blue Jay,0.671875,jay/crow,0.714844,review,NaN,cv,0.003906,capture_20261014T215410_546302Z_69313778 captu...
68,esp32cam-b8aa6d0d5020/capture_20261117T164210_...,Tufted Titmouse,48,White-breasted Nuthatch,0.660645,nuthatch,0.666911,review,NaN,cv,0.025716,capture_20261117T164819_604524Z_4be9f2e9 captu...
26,esp32cam-b8aa6d0d5020/capture_20261028T154956_...,White-breasted Nuthatch,19,House Sparrow,0.653577,sparrow-like,0.681949,review,NaN,cv,0.074836,capture_20261028T154956_488131Z_b6d44b73 captu...
32,esp32cam-b8aa6d0d5020/capture_20261028T191144_...,Black-capped Chickadee,1,Blue Jay,0.632812,jay/crow,0.644531,review,NaN,cv,0.019531,capture_20261028T191144_602119Z_b6c11a79
56,esp32cam-b8aa6d0d5020/capture_20261117T142921_...,White-breasted Nuthatch,7,House Sparrow,0.626674,sparrow-like,0.660714,review,NaN,cv,0.145089,capture_20261117T143357_370093Z_ca01c794 captu...
14,esp32cam-389e8af4e9d4/capture_20261028T180253_...,Blue Jay,12,House Sparrow,0.604492,sparrow-like,0.667318,review,NaN,cv,0.141927,capture_20261028T180448_747250Z_bc16cca3 captu...


final model trained in 5.2 min; files in /mnt/c/Users/Joe/Desktop/ESP_Classifier/esp32_Classifier/Models/runs/esp32_species_v4_1_cv_20261119T165807Z


## 13. Summary and deployment notes


In [13]:
fmt = lambda t: "never (always review)" if t > 1 else f"{t:.3f}"
summary = {
    "run": RUN_NAME, "created_utc": created_utc.isoformat(), "quick_run": QUICK_RUN, "folds": FOLDS,
    "images": len(records), "visits": len(visits), "localizer": chosen_localizer,
    "localisation_rates": match_table.loc["ALL"].to_dict(),
    "cv": {a: {"auto_share": i["auto_share"], "visit_macro_recall": i["visit_macro_recall"], "t_species": i["t_species"],
               "t_group": i["t_group"], "tracks": len(i["tracks"]), "minutes": cv_minutes[a]} for a, i in cv_info.items()},
    "selected_arm": best_arm,
    "out_of_fold_automation": cv_automation[cv_automation["arm"].eq(best_arm)].to_dict("records"),
    "test": test_results.to_dict("records"), "test_automation": test_automation.to_dict("records"),
    "config": {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, bool, list, tuple, dict))},
}
write_json(RUN_DIR / "run_summary.json", summary)
deploy = f"""# Deploying {RUN_NAME}

Files (keep together): model_uint8.tflite, labels.txt, species_contract.json

    python server/raspberry_pi_server.py <camera> \\
        --is-bird-model   <isbird run>/isbird/model_uint8.tflite \\
        --localizer-model <v3 run>/selected_model/esp32_detector_int8.tflite \\
        --species-model   {RUN_DIR / 'model_uint8.tflite'} \\
        --visit-gap-seconds {VISIT_GAP_SECONDS}

Auto-labelling (chosen on {FOLDS}-fold out-of-fold visit tracks, target precision {AUTO_TARGET_PRECISION}):
- species if the top species is one of {sorted(allowed_auto & set(classes))} and its mean probability >= {fmt(t_species)}
- otherwise group if the top group's summed probability >= {fmt(t_group)}
- otherwise review.
Run the first deployment in shadow mode: check visits.jsonl decisions as you label, then retrain.
"""
(RUN_DIR / "DEPLOY.md").write_text(deploy, encoding="utf-8")
print(deploy)


# Deploying esp32_species_v4_1_cv_20261119T165807Z

Files (keep together): model_uint8.tflite, labels.txt, species_contract.json

    python server/raspberry_pi_server.py <camera> \
        --is-bird-model   <isbird run>/isbird/model_uint8.tflite \
        --localizer-model <v3 run>/selected_model/esp32_detector_int8.tflite \
        --species-model   /mnt/c/Users/Joe/Desktop/ESP_Classifier/esp32_Classifier/Models/runs/esp32_species_v4_1_cv_20261119T165807Z/model_uint8.tflite \
        --visit-gap-seconds 120

Auto-labelling (chosen on 4-fold out-of-fold visit tracks, target precision 0.95):
- species if the top species is one of ['Black-capped Chickadee', 'Blue Jay', 'House Sparrow', 'Mourning Dove', 'Northern Cardinal', 'Song Sparrow', 'White-breasted Nuthatch'] and its mean probability >= 0.770
- otherwise group if the top group's summed probability >= never (always review)
- otherwise review.
Run the first deployment in shadow mode: check visits.jsonl decisions as you label, then r